# MLE ile Akilli Sehir Planlamasi

Bu notebook, Poisson dagilimi icin Maximum Likelihood Estimation (MLE) adimlarini sayisal olarak uygulamak, gorsellestirmek ve outlier etkisini incelemek icin hazirlanmistir.

## Kutuphaneler ve Veri Yukleme

Bu bolumde odevde istenen kutuphaneler import edilir ve trafik verisi tanimlanir.

In [ ]:
import numpy as np
import scipy.optimize as opt
from scipy.stats import poisson
import matplotlib.pyplot as plt

# Gozlemlenen trafik verisi: 1 dakikada gecen arac sayisi
traffic_data = np.array([12, 15, 10, 8, 14, 11, 13, 16, 9, 12, 11, 14, 10, 15])

print("Veri seti:", traffic_data)
print("Gozlem sayisi:", len(traffic_data))
print("Toplam arac sayisi:", np.sum(traffic_data))

## Bolum 2: Sayisal (Numerical) MLE

Bu bolumde Poisson dagilimi icin negatif log-likelihood fonksiyonu yazilir ve `scipy.optimize` ile minimum yapan `lambda` degeri bulunur.

In [ ]:
def negative_log_likelihood(lam, data):
    """
    Poisson dagilimi icin negatif log-likelihood hesaplar.

    Not:
    log(k!) terimi lambda'ya bagli olmadigi icin optimizasyonda sabit kabul
    edilir ve ihmal edilebilir.
    """
    # scipy.optimize bazen parametreyi dizi olarak gonderir; sayiya ceviriyoruz.
    lam = float(np.atleast_1d(lam)[0])

    # Poisson parametresi pozitif olmali; aksi durumda fonksiyon anlamsizdir.
    if lam <= 0:
        return np.inf

    n = len(data)
    sum_k = np.sum(data)

    # l(lambda) = -n*lambda + (sum k_i) * log(lambda) - sabit
    # Negatif log-likelihood minimizasyonda kullanilir.
    nll = n * lam - sum_k * np.log(lam)
    return nll


def estimate_lambda_numerical(data, initial_guess=1.0):
    """NLL fonksiyonunu minimize ederek sayisal lambda tahmini yapar."""
    result = opt.minimize(
        negative_log_likelihood,
        x0=np.array([initial_guess]),
        args=(data,),
        bounds=[(0.001, None)],
        method="L-BFGS-B",
    )
    return result


# Analitik cozum: Poisson icin MLE, verinin ortalamasidir.
lambda_analytic = np.mean(traffic_data)

# Sayisal cozum: negatif log-likelihood fonksiyonunu minimize ediyoruz.
result = estimate_lambda_numerical(traffic_data, initial_guess=1.0)
lambda_numerical = result.x[0]

print("=== ORIJINAL VERI SETI ===")
print(f"Analitik tahmin (ortalama): {lambda_analytic:.6f}")
print(f"Sayisal tahmin (MLE lambda): {lambda_numerical:.6f}")
print(f"Minimum NLL degeri: {result.fun:.6f}")
print(f"Optimizasyon basarili mi?: {result.success}")

## Bolum 3: Model Karsilastirma ve Gorsellestirme

Bu bolumde bulunan `lambda` degeri ile Poisson PMF cizilir ve gercek veri histogrami ile ayni grafikte karsilastirilir.

In [ ]:
def plot_histogram_and_pmf(data, lam, title):
    """Veri histogramini ve Poisson PMF egrisini ayni grafikte cizer."""
    x_values = np.arange(0, max(data) + 5)
    pmf_values = poisson.pmf(x_values, mu=lam)

    plt.figure(figsize=(10, 6))

    # Ayrik veri oldugu icin histogram kutulari tam sayi merkezli secilir.
    bins = np.arange(data.min(), data.max() + 2) - 0.5
    plt.hist(
        data,
        bins=bins,
        density=True,
        alpha=0.6,
        color="skyblue",
        edgecolor="black",
        label="Gercek veri histogrami",
    )
    plt.plot(
        x_values,
        pmf_values,
        "o-",
        color="darkred",
        linewidth=2,
        label=f"Poisson PMF (lambda = {lam:.3f})",
    )

    # Odevde istendigi gibi eksen isimleri, baslik ve legend eklenir.
    plt.xlabel("1 dakikadaki arac sayisi")
    plt.ylabel("Olasilik / Yogunluk")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_histogram_and_pmf(
    traffic_data,
    lambda_numerical,
    "Orijinal Veri ve Poisson PMF Karsilastirmasi",
)

print("Yorum: Histogram 10-15 araliginda yogunlasmaktadir. Tahmin edilen lambda degeri de bu bolgeye yakin oldugu icin Poisson modeli veriye makul bir uyum gostermektedir.")

## Bolum 4: Gercek Hayat Senaryosu - Outlier Analizi

Bu bolumde veri setine yanlislikla kaydedilen `200` degerli bir aykiri gozlem eklenir ve MLE tahmininin bu durumdan nasil etkilendigi incelenir.

In [ ]:
# Veri setine aykiri bir gozlem ekliyoruz.
traffic_data_outlier = np.append(traffic_data, 200)

# Outlier eklendikten sonraki analitik ve sayisal tahminleri hesapliyoruz.
lambda_analytic_outlier = np.mean(traffic_data_outlier)
result_outlier = estimate_lambda_numerical(traffic_data_outlier, initial_guess=1.0)
lambda_numerical_outlier = result_outlier.x[0]

print("=== OUTLIER EKLENMIS VERI SETI ===")
print("Yeni veri seti:", traffic_data_outlier)
print(f"Analitik tahmin (ortalama): {lambda_analytic_outlier:.6f}")
print(f"Sayisal tahmin (MLE lambda): {lambda_numerical_outlier:.6f}")
print(f"Optimizasyon basarili mi?: {result_outlier.success}")

In [ ]:
plot_histogram_and_pmf(
    traffic_data_outlier,
    lambda_numerical_outlier,
    "Outlier Sonrasi Veri ve Poisson PMF Karsilastirmasi",
)

print("Yorum: Tek bir 200 degerli aykiri gozlem, lambda tahminini ciddi bicimde yukari cekmistir. Bu durum MLE'nin ortalamaya esit olmasi nedeniyle aykiri degerlere hassas oldugunu gosterir.")
print("Gercek hayatta boyle bir hata, belediyenin trafik yogunlugunu oldugundan fazla tahmin etmesine ve yanlis planlama kararlarina yol acabilir.")